# Demo: On-Device Text Summarizer
## **Objectives:**
* Environment Prep (Colab-safe)- Pin numpy==1.26.4 and auto-restart runtime.
Install transformers, accelerate, bitsandbytes, huggingface_hub (no PyTorch reinstall).

* Load Summarizer (Baseline)- Use sshleifer/distilbart-cnn-12-6 with pipeline("summarization").
Run on a sample paragraph and print latency.

* Model Card Quick Check Use HfApi().model_info(MODEL_ID).
Read license, pipeline_tag, and tags to confirm suitability.

* 8-bit Quantization (GPU)- Reload the same model with BitsAndBytesConfig(load_in_8bit=True).
Compare GPU memory via nvidia-smi and time per summary before/after.

* User Text Tryout Accept a paragraph string input from the user.
Summarize it and print the summary


In [ ]:
#@title Pin NumPy and **auto‑restart** (run first)
!pip -q install --upgrade 'numpy==1.26.4'
import os, sys
print('Pinned NumPy to 1.26.4; restarting runtime to load correct binaries...')
os.kill(os.getpid(), 9)  # Colab-friendly hard restart


In [1]:
#@title Step 1: Install libraries (Transformer stack only)
!pip -q install transformers==4.42.4 accelerate==0.33.0 bitsandbytes==0.43.1 huggingface_hub==0.24.6 langdetect==1.0.9
import torch, subprocess
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        name = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
    except Exception:
        name = 'Unknown GPU'
    print('GPU:', name)
else:
    print('Running on CPU — the demo still works (8-bit step will be skipped).')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 796.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
  

## 1) Baseline: Load a small summarizer and run it
We use **DistilBART CNN** (`sshleifer/distilbart-cnn-12-6`).

In [2]:
from transformers import pipeline
import time
MODEL_ID = 'sshleifer/distilbart-cnn-12-6'
summarizer = pipeline('summarization', model=MODEL_ID, device=0 if torch.cuda.is_available() else -1)
sample_text = (
    "Open-source large language models let small teams build features fast. "
    "With Hugging Face, you can prototype summarization in minutes, then optimize with quantization to reduce cost and latency."
)
t0 = time.time()
baseline = summarizer(sample_text, max_length=90, min_length=30, do_sample=False)[0]['summary_text']
t1 = time.time()
print('Baseline summary:', baseline)
print('Time (s):', round(t1 - t0, 3))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Your max_length is set to 90, but your input_length is only 39. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)


Baseline summary:  Hugging Face is an open-source language model that lets small teams build features fast . It lets you prototype summarization in minutes, then optimize with quantization to reduce cost and latency .
Time (s): 7.564


## 2) Quick peek at the model card

In [3]:
from huggingface_hub import HfApi
api = HfApi()
info = api.model_info(MODEL_ID)
print('Model:', info.modelId)
print('License:', getattr(info, 'license', 'unknown'))
print('Pipeline tag:', getattr(info, 'pipeline_tag', 'n/a'))
print('Tags:', ', '.join(info.tags or []))
print('More details:', f'https://huggingface.co/{MODEL_ID}')


Model: sshleifer/distilbart-cnn-12-6
License: unknown
Pipeline tag: summarization
Tags: transformers, pytorch, jax, rust, bart, text2text-generation, summarization, en, dataset:cnn_dailymail, dataset:xsum, license:apache-2.0, endpoints_compatible, region:us
More details: https://huggingface.co/sshleifer/distilbart-cnn-12-6


## 3) 8-bit quantization with bitsandbytes (GPU)

In [4]:
import gc, subprocess
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig, pipeline as hf_pipeline

def gpu_mem():
    if not torch.cuda.is_available():
        return 'CUDA not available'
    try:
        out = subprocess.check_output('nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits', shell=True).decode().strip()
        used, total = [x.strip() for x in out.split(',')]
        return f"{used} / {total} MB"
    except Exception:
        return 'nvidia-smi not accessible'

print('GPU mem BEFORE:', gpu_mem())
if torch.cuda.is_available():
    del summarizer
    gc.collect()
    torch.cuda.empty_cache()

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
    model_8 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map='auto')
    summarizer_8 = hf_pipeline('summarization', model=model_8, tokenizer=tok, device=0)

    print('GPU mem AFTER 8-bit load:', gpu_mem())
    import time
    t0 = time.time()
    out = summarizer_8(sample_text, max_length=90, min_length=30, do_sample=False)[0]['summary_text']
    t1 = time.time()
    print('8-bit summary:', out)
    print('Time (s):', round(t1 - t0, 3))
else:
    print('Skipping 8-bit quantization: GPU not available.')


GPU mem BEFORE: CUDA not available
Skipping 8-bit quantization: GPU not available.


In [5]:
import torch
from transformers import pipeline

MODEL_ID = "sshleifer/distilbart-cnn-12-6"

# Use GPU if available
device = 0 if torch.cuda.is_available() else -1
summarizer = pipeline("summarization", model=MODEL_ID, device=device)

paragraph = """
Open-source large language models let small teams build features quickly.
With Hugging Face, you can prototype summarization, translation, and Q&A in minutes.
Optimizations like quantization reduce latency and memory, making deployment cheaper.
"""

# Run the summary
result = summarizer(paragraph, max_length=90, min_length=25, do_sample=False)[0]["summary_text"]
print("Summary:\n", result)


Your max_length is set to 90, but your input_length is only 55. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=27)


Summary:
  Hugging Face lets you prototype summarization, translation, and Q&A in minutes . Open-source large language models let small teams build features quickly .
